# CACS prediction: snapshot cohort - all five models

Part of the thesis *Machine Learning for the Prediction of Coronary Artery Calcium Score Using Longitudinal Data* (Kasetsart University).

**Outcome:** CACS of 100 or more. **Input:** `snapshot_730.csv` (1,095 patients, 47 features).

**Steps:** features, 5-fold cross-validation repeated 3 times (the same 15 folds for every model), imputation fitted on the training fold only, five models (logistic regression, random forest, support vector machine, XGBoost, TabPFN), evaluation (accuracy, macro precision, macro recall, macro F1, AUC), ROC curves, and permutation importance.

**Data:** the patient data are not included (see `README.md`). **Runtime:** use a GPU runtime in Colab, because permutation importance for TabPFN is slow. Results are saved after every fold, so if Colab stops, run all cells again and it continues.


In [ ]:
!pip install -q tabpfn xgboost


In [ ]:
import sys, pandas, numpy, sklearn, xgboost, tabpfn, matplotlib
print(sys.version)
for m in [pandas, numpy, sklearn, xgboost, tabpfn, matplotlib]:
    print(m.__name__, m.__version__)


In [ ]:
# ---- TabPFN token ----
# TabPFN needs a personal access token. Do NOT write the token in this notebook.
# In Colab, add it once under "Secrets" (key icon on the left) with the name TABPFN_TOKEN.
import os
try:
    from google.colab import userdata
    os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
except Exception:
    pass                                   # outside Colab: set TABPFN_TOKEN as an environment variable
print("token set:", bool(os.environ.get("TABPFN_TOKEN")))


In [ ]:
# ---- data file ----
# The patient data are NOT included in this repository (see "Data availability" in README.md).
# In Colab, upload snapshot_730.csv when asked; elsewhere, put it in the same folder as this notebook.
import glob
if not os.path.exists("snapshot_730.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError("snapshot_730.csv not found")
print("csv files here:", glob.glob("*.csv"))


In [ ]:
# ============ MODELLING: SNAPSHOT FEATURES ============
# Procedure:
#   - 5-fold cross-validation repeated 3 times (the same 15 folds for every model)
#   - IterativeImputer (ExtraTrees) fitted on the training fold only
#   - fixed model settings (no tuning), threshold 0.5
#   - accuracy, macro precision, macro recall, macro F1, AUC
#   - permutation importance on the held-out patients of each fold
# Results are saved after every fold; if Colab stops, run the cells again and it continues.
import os, time
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesRegressor
from sklearn.svm import SVC
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.utils.class_weight import compute_sample_weight

DATA_DIR = "."
USE_TABPFN = True            # include TabPFN as a fifth model
TABPFN_IMPORTANCE = True     # permutation importance for TabPFN too (slow: use a GPU runtime)

CV_OUT = rf"{DATA_DIR}/cv_results_snapshot.csv"         # metrics, one row per fold x model
PRED_OUT = rf"{DATA_DIR}/cv_predictions_snapshot.csv"   # each patient's predicted probability
IMP_OUT = rf"{DATA_DIR}/cv_importance_snapshot.csv"     # permutation importance per fold x model

CV_MEAS = ['fpg', 'hba1c', 'chol', 'hdl', 'ldl', 'tg', 'egfr', 'ua', 'wt', 'ht', 'bmi', 'sbp', 'dbp']
snap = pd.read_csv(rf"{DATA_DIR}/snapshot_730.csv", index_col=0)

# SNAPSHOT FEATURES: age, sex, the most recent value of each variable, days since each
# measurement, number of measurements, and the derived ratios (47 features)
CV_RATIOS = ['nonhdl', 'chol_hdl_ratio', 'tg_hdl_ratio', 'ldl_hdl_ratio', 'pulse_pressure', 'map']
CV_FEATS = (['age', 'sex_male'] + CV_MEAS
            + [f'{m}_{s}' for m in CV_MEAS for s in ('lag', 'n')] + CV_RATIOS)
CV_FEATS = [c for c in CV_FEATS if c in snap.columns]

X_cv = snap[CV_FEATS]
y_cv = snap['y2'].astype(int).to_numpy()
cv_folds = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
                .split(X_cv, snap['y4']))          # y4 is used ONLY to build the folds
print("snapshot:", X_cv.shape, "| folds:", len(cv_folds), "| expected: (1095, 47)")

In [ ]:
def cv_models():
    m = {
        'Logistic regression': Pipeline([('scaler', StandardScaler()),
                                         ('clf', LogisticRegression(C=0.1, max_iter=5000,
                                                                    class_weight='balanced', random_state=42))]),
        'Random forest': RandomForestClassifier(n_estimators=500, min_samples_leaf=5, max_features='sqrt',
                                                class_weight='balanced_subsample', random_state=42, n_jobs=-1),
        'Support vector machine': Pipeline([('scaler', StandardScaler()),
                                            ('clf', SVC(C=2.0, gamma='scale', probability=True,
                                                        class_weight='balanced', random_state=42))]),
    }
    try:
        from xgboost import XGBClassifier
        m['XGBoost'] = XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=3, min_child_weight=5,
                                     subsample=0.8, colsample_bytree=0.6, reg_lambda=5.0, reg_alpha=0.5,
                                     tree_method='hist', random_state=42, n_jobs=-1, eval_metric='logloss')
    except ImportError:
        print("xgboost not installed - skipping it")
    if USE_TABPFN:
        from tabpfn import TabPFNClassifier
        try:
            m['TabPFN'] = TabPFNClassifier(device='auto', random_state=42)
        except TypeError:                     # older versions have no random_state
            m['TabPFN'] = TabPFNClassifier(device='auto')
    return m


def cv_impute(Xtr, Xte):
    keep = [c for c in Xtr.columns if Xtr[c].notna().any()]
    Xtr, Xte = Xtr[keep], Xte[keep]
    imp = IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=100, random_state=42),
                           max_iter=5, random_state=42, initial_strategy='median')
    imp.fit(Xtr)
    cols = list(imp.get_feature_names_out())
    a = pd.DataFrame(imp.transform(Xtr), columns=cols, index=Xtr.index)
    b = pd.DataFrame(imp.transform(Xte), columns=cols, index=Xte.index)
    med = Xtr.median(numeric_only=True)
    return a.fillna(med).fillna(0), b.fillna(med).fillna(0)


def finished(path):
    return set(pd.read_csv(path).fold) if os.path.exists(path) else set()

done = finished(CV_OUT) & finished(PRED_OUT) & finished(IMP_OUT)   # a fold is done only if all three were saved
for k, (tr, te) in enumerate(cv_folds):
    if k in done:
        continue
    t0 = time.time()
    Xtr, Xte = cv_impute(X_cv.iloc[tr].copy(), X_cv.iloc[te].copy())
    rows, preds, imps = [], [], []
    for name, model in cv_models().items():
        if name == 'XGBoost':
            model.fit(Xtr, y_cv[tr], sample_weight=compute_sample_weight('balanced', y_cv[tr]))
        else:
            model.fit(Xtr, y_cv[tr])
        p = model.predict_proba(Xte)[:, 1]
        pred = (p >= 0.5).astype(int)
        rows.append({'model': name, 'fold': k,
                     'accuracy': accuracy_score(y_cv[te], pred),
                     'macro_precision': precision_score(y_cv[te], pred, average='macro', zero_division=0),
                     'macro_recall': recall_score(y_cv[te], pred, average='macro', zero_division=0),
                     'macro_f1': f1_score(y_cv[te], pred, average='macro'),
                     'auc': roc_auc_score(y_cv[te], p)})
        preds.append(pd.DataFrame({'model': name, 'fold': k, 'HN': X_cv.index[te], 'y': y_cv[te], 'p': p}))
        # permutation importance: drop in AUC when one feature is shuffled in the held-out patients
        if name != 'TabPFN' or TABPFN_IMPORTANCE:
            r = permutation_importance(model, Xte, y_cv[te], n_repeats=5, random_state=42, scoring='roc_auc')
            imps.append(pd.DataFrame({'model': name, 'fold': k, 'feature': Xte.columns,
                                      'importance': r.importances_mean}))
    pd.DataFrame(rows).to_csv(CV_OUT, mode='a', index=False, header=not os.path.exists(CV_OUT))
    pd.concat(preds).to_csv(PRED_OUT, mode='a', index=False, header=not os.path.exists(PRED_OUT))
    pd.concat(imps).to_csv(IMP_OUT, mode='a', index=False, header=not os.path.exists(IMP_OUT))
    print(f"  fold {k+1:2d}/15  ({time.time()-t0:.0f}s)  " +
          ' | '.join(f"{r['model'].split()[0]} {r['auc']:.3f}" for r in rows), flush=True)

In [ ]:
# ---- model performance: mean over the 15 folds (Table 10) ----
cv = pd.read_csv(CV_OUT).drop_duplicates(['model', 'fold'])
metrics = ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'auc']
cv_table = cv.groupby('model')[metrics].mean().sort_values('auc', ascending=False)
cv_table.columns = ['Accuracy', 'Macro precision', 'Macro recall', 'Macro F1', 'AUC']
cv_table['SD of AUC'] = cv.groupby('model').auc.std()          # SD over the 15 folds
print(f"{cv.fold.nunique()} folds")
print(cv_table.round(3).to_string())
cv_table.round(3).to_csv(rf"{DATA_DIR}/table10_snapshot.csv")

In [ ]:
# ---- ROC curves from cross-validation ----
# One ROC curve per fold, using only the patients that fold's model never saw;
# the curves are averaged over the 15 folds. The shaded band is +/- 1 SD for
# the best model. Legend AUC = mean +/- SD over the folds, as in the performance table.
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

preds = pd.read_csv(PRED_OUT).drop_duplicates(['model', 'fold', 'HN'])
grid = np.linspace(0, 1, 201)
plt.figure(figsize=(8, 6.5))
for i, m in enumerate(cv_table.index):                      # ordered by AUC
    tprs, aucs = [], []
    for k, g in preds[preds.model == m].groupby('fold'):
        fpr, tpr, _ = roc_curve(g.y, g.p)
        t = np.interp(grid, fpr, tpr); t[0] = 0.0
        tprs.append(t); aucs.append(auc(fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    plt.plot(grid, mean_tpr, lw=2 if i == 0 else 1.5,
             label=f"{m} (AUC {np.mean(aucs):.3f} \u00b1 {np.std(aucs, ddof=1):.3f})")
    if i == 0:
        sd = np.std(tprs, axis=0, ddof=1)
        plt.fill_between(grid, np.clip(mean_tpr - sd, 0, 1), np.clip(mean_tpr + sd, 0, 1), alpha=0.15)
plt.plot([0, 1], [0, 1], '--', color='gray', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC - snapshot cohort, CACS >= 100 (5-fold cross-validation, 3 repeats)')
plt.legend(loc='lower right', fontsize=9); plt.tight_layout()
plt.savefig(rf"{DATA_DIR}/roc_cv_snapshot.png", dpi=150)
plt.show()

In [ ]:
# ---- feature importance: permutation importance, mean over the 15 folds ----
# Importance = drop in AUC when a feature is shuffled in the held-out patients,
# averaged over the folds, negative values set to zero, and expressed as % of each model's total.
TOP_N = 12
imp = pd.read_csv(IMP_OUT).drop_duplicates(['model', 'fold', 'feature'])
mean_imp = imp.groupby(['model', 'feature']).importance.mean().clip(lower=0).reset_index()
mean_imp['pct'] = 100 * mean_imp.importance / mean_imp.groupby('model').importance.transform('sum')

models = [m for m in cv_table.index if m in set(mean_imp.model)]   # all five models
fig, axes = plt.subplots(int(np.ceil(len(models) / 2)), 2, figsize=(13, 3.6 * int(np.ceil(len(models) / 2))))
axes = np.atleast_1d(axes).ravel()
for ax, m in zip(axes, models):
    top = mean_imp[mean_imp.model == m].nlargest(TOP_N, 'pct').iloc[::-1]
    ax.barh(top.feature, top.pct)
    ax.set_xlabel('% of total permutation importance'); ax.set_title(m, fontsize=11)
    ax.tick_params(axis='y', labelsize=8)
for ax in axes[len(models):]:
    ax.axis('off')
fig.suptitle('Permutation importance by model (5-fold cross-validation, 3 repeats)', y=1.002)
fig.tight_layout()
plt.savefig(rf"{DATA_DIR}/importance_cv_snapshot.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- top 3 features per model, with % (Table 12) ----
rows = []
for m in models:
    top3 = mean_imp[mean_imp.model == m].nlargest(3, 'pct')
    row = {'Model': m}
    for i, r in enumerate(top3.itertuples(), start=1):
        row[f'#{i}'] = f"{r.feature} ({r.pct:.1f}%)"
    row['top 3 combined'] = f"{top3.pct.sum():.1f}%"
    rows.append(row)
top3_table = pd.DataFrame(rows)
print(top3_table.to_string(index=False))
top3_table.to_csv(rf"{DATA_DIR}/top3_features_snapshot.csv", index=False)